# LevenbergMarquardtParams

`LevenbergMarquardtParams` combines the common nonlinear-optimizer settings with damping controls specific to Levenberg–Marquardt. It determines how the optimizer moves between Gauss–Newton and gradient-descent behavior.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/nonlinear/doc/LevenbergMarquardtParams.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [ ]:
import gtsam
import numpy as np
from gtsam.symbol_shorthand import L, V, X

## Defaults and construction

The ordinary constructor uses GTSAM defaults. `CeresDefaults()` and `LegacyDefaults()` provide named parameter sets for compatibility with those conventions.

In [ ]:
params = gtsam.LevenbergMarquardtParams.CeresDefaults()
print("maximum iterations:", params.getMaxIterations())
print("initial lambda:", params.getlambdaInitial())
print("lambda factor:", params.getlambdaFactor())

## Damping controls

`lambdaInitial`, lower/upper bounds, and `lambdaFactor` control damping adaptation. `setDiagonalDamping(True)` scales damping by the Hessian diagonal; `setUseFixedLambdaFactor()` selects fixed multiplicative updates.

In [ ]:
params.setlambdaInitial(1e-3)
params.setlambdaLowerBound(1e-8)
params.setlambdaUpperBound(1e5)
params.setlambdaFactor(5.0)
params.setDiagonalDamping(True)
params.setVerbosityLM("SILENT")

## Using the parameters

Common inherited methods select convergence tolerances, iteration limits, ordering, and linear solver. The optimizer copies the parameter object at construction.

In [ ]:
graph = gtsam.NonlinearFactorGraph()
model = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.1, 0.1, 0.05]))
graph.add(gtsam.PriorFactorPose2(X(0), gtsam.Pose2(1.0, 2.0, 0.3), model))
initial = gtsam.Values()
initial.insert(X(0), gtsam.Pose2(0.0, 0.0, 0.0))

optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial, params)
result = optimizer.optimize()
print("optimized pose:", result.atPose2(X(0)))
assert graph.error(result) < graph.error(initial)

## Diagnosing damping behavior

If lambda repeatedly reaches its upper bound, inspect scaling, initialization, and factor Jacobians instead of simply increasing the bound. If convergence is smooth but slow, compare diagonal damping and the named default sets, while keeping the same stopping tolerances for a fair comparison.

## Source

[`LevenbergMarquardtParams.h`](../LevenbergMarquardtParams.h)

## AI assistance caveat

AI was used to help draft this documentation, and inaccuracies could be present.